In [1]:
"""
sentiment_scoring.py
────────────────────
Runs FinBERT sentiment scoring on news_master.csv.
Produces:
  - news_scored.csv     : article-level scores
  - sentiment_daily.csv : one row per ticker per trading day

Run on Google Colab with T4 GPU for best performance.
Upload news_master.csv before running.
"""

import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import time

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "ProsusAI/finbert"
BATCH_SIZE  = 32
MAX_LENGTH  = 512
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"── FINBERT SENTIMENT SCORING ────────────────────────────")
print(f"  Device     : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU        : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ── Dataset class ─────────────────────────────────────────────────────────────
class ArticleDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


# ── Load model ────────────────────────────────────────────────────────────────
def load_finbert():
    print(f"\n  Loading FinBERT model ({MODEL_NAME})...")
    print(f"  First run downloads ~440MB — subsequent runs use cache")
    t0        = time.time()
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    model     = BertForSequenceClassification.from_pretrained(MODEL_NAME)
    model     = model.to(DEVICE)
    model.eval()
    print(f"  ✓ Model loaded in {time.time() - t0:.1f}s")
    label_map = {0: "positive", 1: "negative", 2: "neutral"}
    return tokenizer, model, label_map


# ── Score in batches ──────────────────────────────────────────────────────────
def score_finbert(texts: list, tokenizer, model, label_map: dict) -> pd.DataFrame:
    """
    Score a list of texts with FinBERT.
    Returns DataFrame with columns:
        finbert_label, finbert_positive, finbert_negative,
        finbert_neutral, finbert_score
    finbert_score = positive_prob - negative_prob (ranges -1 to +1)
    """
    dataset    = ArticleDataset(texts, tokenizer, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_probs = []
    t0        = time.time()

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)

            done    = min((i + 1) * BATCH_SIZE, len(texts))
            elapsed = time.time() - t0
            speed   = done / elapsed if elapsed > 0 else 0
            eta     = (len(texts) - done) / speed if speed > 0 else 0
            print(f"\r  Scoring: {done:>6}/{len(texts)} articles "
                  f"| {speed:.0f} art/s "
                  f"| ETA: {eta:.0f}s   ", end="", flush=True)

    print()
    all_probs = np.vstack(all_probs)

    results = pd.DataFrame({
        "finbert_positive" : all_probs[:, 0].round(4),
        "finbert_negative" : all_probs[:, 1].round(4),
        "finbert_neutral"  : all_probs[:, 2].round(4),
    })

    results["finbert_label"] = [label_map[i] for i in all_probs.argmax(axis=1)]
    results["finbert_score"] = (
        results["finbert_positive"] - results["finbert_negative"]
    ).round(4)

    return results


# ── Aggregate to daily sentiment ──────────────────────────────────────────────
def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse article-level scores into one row per ticker per trading day.
    Output columns:
        ticker, trading_date, avg_finbert, std_finbert,
        bullish_ratio, bearish_ratio, neutral_ratio,
        avg_prescore, article_count
    """
    def agg(g):
        return pd.Series({
            "avg_finbert"   : round(float(g["finbert_score"].mean()), 4),
            "std_finbert"   : round(float(g["finbert_score"].std()), 4) if len(g) > 1 else 0.0,
            "bullish_ratio" : round(float((g["finbert_label"] == "positive").mean()), 4),
            "bearish_ratio" : round(float((g["finbert_label"] == "negative").mean()), 4),
            "neutral_ratio" : round(float((g["finbert_label"] == "neutral").mean()), 4),
            "avg_prescore"  : round(float(g["prescore"].mean()), 4) if g["prescore"].notna().any() else None,
            "article_count" : len(g),
        })

    daily = (df.groupby(["ticker", "trading_date"])
               .apply(agg, include_groups=False)
               .reset_index())

    daily["trading_date"] = pd.to_datetime(daily["trading_date"])
    daily = daily.sort_values(["ticker", "trading_date"]).reset_index(drop=True)
    return daily


# ── Main ──────────────────────────────────────────────────────────────────────
def run_sentiment_scoring(news_path="news_master.csv"):

    # ── Load ──────────────────────────────────────────────────────────────────
    df = pd.read_csv(news_path)
    df["trading_date"] = pd.to_datetime(df["trading_date"])
    print(f"\n  Loaded {len(df):,} articles from {news_path}")
    print(f"  Tickers: {sorted(df['ticker'].unique().tolist())}")

    # ── Prepare texts ─────────────────────────────────────────────────────────
    texts = df["text_for_sentiment"].fillna("").str[:1000].tolist()

    # ── Load model ────────────────────────────────────────────────────────────
    tokenizer, model, label_map = load_finbert()

    # ── Score ─────────────────────────────────────────────────────────────────
    print(f"\n  Scoring {len(texts):,} articles in batches of {BATCH_SIZE}...")
    t0      = time.time()
    scores  = score_finbert(texts, tokenizer, model, label_map)
    elapsed = time.time() - t0
    print(f"  ✓ Scoring complete in {elapsed:.1f}s "
          f"({len(texts)/elapsed:.0f} articles/sec)")

    # ── Attach scores ─────────────────────────────────────────────────────────
    df = pd.concat([df.reset_index(drop=True), scores], axis=1)

    # ── Article-level stats ───────────────────────────────────────────────────
    print(f"\n── ARTICLE-LEVEL RESULTS ────────────────────────────────")
    print(f"  finbert_score stats:")
    print(f"    Mean   : {df['finbert_score'].mean():.4f}")
    print(f"    Std    : {df['finbert_score'].std():.4f}")
    print(f"    Min    : {df['finbert_score'].min():.4f}")
    print(f"    Max    : {df['finbert_score'].max():.4f}")

    print(f"\n  Label distribution (all articles):")
    for label, cnt in df["finbert_label"].value_counts().items():
        pct = cnt / len(df) * 100
        bar = "█" * int(pct / 3)
        print(f"    {label:<10} {cnt:>6,} ({pct:4.1f}%) {bar}")

    print(f"\n  Per-ticker breakdown:")
    for ticker, grp in df.groupby("ticker"):
        print(f"\n  {ticker}:")
        print(f"    Articles          : {len(grp):,}")
        print(f"    Avg finbert_score : {grp['finbert_score'].mean():.4f}")
        for label, cnt in grp["finbert_label"].value_counts().items():
            pct = cnt / len(grp) * 100
            print(f"    {label:<10} {cnt:>5,} ({pct:.1f}%)")

    # ── Most positive and negative articles ───────────────────────────────────
    print(f"\n── MOST POSITIVE ARTICLE ────────────────────────────────")
    top = df.loc[df["finbert_score"].idxmax()]
    print(f"  Ticker  : {top['ticker']}")
    print(f"  Date    : {top['trading_date']}")
    print(f"  Score   : {top['finbert_score']}")
    print(f"  Title   : {top['title'][:100]}")

    print(f"\n── MOST NEGATIVE ARTICLE ────────────────────────────────")
    bot = df.loc[df["finbert_score"].idxmin()]
    print(f"  Ticker  : {bot['ticker']}")
    print(f"  Date    : {bot['trading_date']}")
    print(f"  Score   : {bot['finbert_score']}")
    print(f"  Title   : {bot['title'][:100]}")

    # ── Aggregate to daily ────────────────────────────────────────────────────
    print(f"\n── DAILY AGGREGATION ────────────────────────────────────")
    daily = aggregate_daily(df)
    print(f"  Output : {len(daily):,} rows (one per ticker per trading day)")
    print(f"  Columns: {list(daily.columns)}")

    # ── Sample output for AAPL and NFLX ──────────────────────────────────────
    print(f"\n  Sample AAPL daily sentiment (last 5 days):")
    aapl = daily[daily["ticker"] == "AAPL"].tail(5)
    print(aapl[["trading_date", "avg_finbert", "bullish_ratio",
                "bearish_ratio", "avg_prescore",
                "article_count"]].to_string(index=False))

    print(f"\n  Sample NFLX daily sentiment (last 5 days):")
    nflx = daily[daily["ticker"] == "NFLX"].tail(5)
    print(nflx[["trading_date", "avg_finbert", "bullish_ratio",
                "bearish_ratio", "avg_prescore",
                "article_count"]].to_string(index=False))

    # ── Coverage check ────────────────────────────────────────────────────────
    print(f"\n  Coverage check (trading days with sentiment):")
    for ticker in sorted(df["ticker"].unique()):
        n = daily[daily["ticker"] == ticker]["trading_date"].nunique()
        print(f"    {ticker}: {n} trading days")

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_csv("news_scored.csv", index=False)
    daily.to_csv("sentiment_daily.csv", index=False)

    print(f"\n── SAVED ────────────────────────────────────────────────")
    print(f"  news_scored.csv     — {len(df):,} rows (article-level + finbert scores)")
    print(f"  sentiment_daily.csv — {len(daily):,} rows (one per ticker per day)")
    print("─" * 55)

    return df, daily


# ── Run ───────────────────────────────────────────────────────────────────────
df_scored, df_daily = run_sentiment_scoring("news_master.csv")

── FINBERT SENTIMENT SCORING ────────────────────────────
  Device     : cuda
  GPU        : Tesla T4
  VRAM       : 15.6 GB

  Loaded 33,338 articles from news_master.csv
  Tickers: ['AAPL', 'AMZN', 'GOOGL', 'JPM', 'META', 'MSFT', 'NFLX', 'NVDA']

  Loading FinBERT model (ProsusAI/finbert)...
  First run downloads ~440MB — subsequent runs use cache


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Model loaded in 7.0s

  Scoring 33,338 articles in batches of 32...
  Scoring:  33338/33338 articles | 53 art/s | ETA: 0s   
  ✓ Scoring complete in 658.1s (51 articles/sec)

── ARTICLE-LEVEL RESULTS ────────────────────────────────
  finbert_score stats:
    Mean   : 0.2664
    Std    : 0.6823
    Min    : -0.9689
    Max    : 0.9460

  Label distribution (all articles):
    positive   17,412 (52.2%) █████████████████
    neutral     8,064 (24.2%) ████████
    negative    7,862 (23.6%) ███████

  Per-ticker breakdown:

  AAPL:
    Articles          : 6,225
    Avg finbert_score : 0.1947
    positive   2,878 (46.2%)
    neutral    1,700 (27.3%)
    negative   1,647 (26.5%)

  AMZN:
    Articles          : 4,044
    Avg finbert_score : 0.2789
    positive   2,126 (52.6%)
    neutral    1,009 (25.0%)
    negative     909 (22.5%)

  GOOGL:
    Articles          : 3,623
    Avg finbert_score : 0.2647
    positive   1,911 (52.7%)
    negative     885 (24.4%)
    neutral      827 (22.8%)